<h1 style="font-size:30px;">Assignment: Implement a CNN for Image Classification</h1>

We started this module from linear regression, and gradually shifted our focus towards multi-class classification preferrably through CNNs.


## <font style="color:rgb(50,120,230)">Marking scheme</font>

### Maximum Points: 30

<div>
    <table>
        <tr style="text-align: center;"><td><h3><strong>Section</strong></h3></td> <td><h3><strong>Problem</strong></h3></td> <td><h3><strong>Points</strong></h3></td></tr>
        <tr><td><h3>1</h3></td> <td><h3>Model Architecture</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>2</h3></td> <td><h3>Accuracy >=92%</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>3</h3></td> <td><h3>Accuracy >=95%</h3></td> <td><h3>10</h3></td> </tr>
    </table>
</div>

## Problem Description

<hr style="border:none; height: 4px; background-color:#D3D3D3" />

### [<font style="color:rgb(50,120,230);text-decoration: underline">1. Implement the CNN Model</font>](#6-Model-Definition-[10-Points])

You are required to implement a CNN model from scratch. The feature extractor should contain atleast the **`Conv2d`**, **`BatchNorm2d`**, and **`ReLU`** layers.


Since we are solving a Binary Classification problem, we can use the **Binary Cross Entropy** loss intead of the usual **Cross Entropy Loss**. Hence, you need to ensure the correct `out_features` in the final **`Linear`** (FC) layer.

**You need to define the model architecture in the class: `BinaryClassifierModel`**. This section carries **`10`** points.

**Note:** For color images you need to use an input shape that has three channels, so that it accepts 3 channel inputs.


### [<font style="color:rgb(50,120,230);text-decoration: underline">2. Model Training and Accuracy</font>](#5-Configuration-[20-Points])

Once you have defined the model, you can train it. To get better accuracy, you need to experiment with the training configuration and even the model architecture. You can check the accuracy by running the training loop.

Here are a few hints on how you can improve the accuracy:
- Train for larger number of epochs
- Try using different learning rates
- Try to add more convolutional layers to the architecture
- Try to add more nodes in the layers.
- Experiment with the training image resolution.


You need to achieve atleast **92% accuracy** to obtain the remaining **`10`** points. To obtain the maximum grade for the assignment you need to achieve atleast **95% accuracy** for which you will be awarded the final **`10`** points.

**PS: Click on the headings above to jump to the respective sections in the notebook.**
<hr style="border:none; height: 4px; background-color:#D3D3D3" />

Since we have over `5000` samples across training and evaluation, building a model from scratch and training it requires significant time. **Leveraging compute resources such as GPUs can be beneficial for faster training.**

For ease of use, the assignment has been divided into two notebooks:

**Notebook 1 (Training)**: This is the training notebook (Colab), which you will use to train your binary classifier model.

You can execute this notebook in Google Colab or Kaggle Kernel to gain access to a GPU machine and prototype faster. Once the desired results are achieved, you should download the model checkpoints.

Access the Colab Notebook: [here](https://colab.research.google.com/drive/1TWchieUs1NqpvmH0X9GL3NhTQJlg5pbM)

**If you use Google Colab or any other compute engine (apart from Vocareum), kindly upload your model under a new folder named `CHECKPOINTS`.**

**Notebook 2 (Evaluation)**: This is the evaluation notebook (the one you are currently viewing in Labs), used to evaluate your model's accuracy. Upload your trained checkpoints and copy-paste the changes you made in the model definition section from Notebook 1. You will also need to update the mean and standard deviation values that you calculated in your training notebook in the `DatasetConfig` section.

> **Note:** Assignments are autograded based on specific **test cases** within individual graded sections. You should not delete any cells or modify the filename of this notebook for the Autograder to function correctly.

> You may only modify sections within  
> ```  
> ### BEGIN SOLUTION  
> ...  
> ### END SOLUTION  
> ```


<hr style="border:none; height: 4px; background-color:#D3D3D3" />




## Table of Contents

- [1 Import Dependencies](#1-Import-Dependencies)
- [2 System Config](#2-System-Config)
- [3 Preprocessing Transforms](#3-Preprocessing-Transforms)
- [4 Create Test Dataloader](#4-Create-Test-Dataloader)
- [5 Configuration](#5-Configuration-[20-Points])
- [6 Model Definition](#6-Model-Definition-[10-Points])
- [7 Evaluation Step](#7-Evaluation-Step)
- [8 Model Prediction on Test Data](#8-Model-Prediction-on-Test-Data)
- [9 Conclusion](#9-Conclusion)

## 1 Import Dependencies

In [27]:
import sys, numpy, importlib
print("Python:", sys.version)
print("NumPy:", numpy.__version__, numpy.__file__)
try:
    import scipy
    print("SciPy:", scipy.__version__, scipy.__file__)
except Exception as e:
    print("SciPy not importable:", e)


Python: 3.10.12 (main, Aug 15 2025, 14:32:43) [GCC 11.4.0]
NumPy: 1.26.4 /home/cv-sse/.local/lib/python3.10/site-packages/numpy/__init__.py
SciPy: 1.11.4 /home/cv-sse/.local/lib/python3.10/site-packages/scipy/__init__.py


In [28]:
import copy
from dataclasses import dataclass
import math
import numpy as np
import os
import random
import time
from typing import List, Union, Tuple
import warnings
from tqdm import tqdm
from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from torchvision.ops import Conv2dNormActivation
from torchvision import datasets, transforms as T
from torchinfo import summary

from torchmetrics import MeanMetric
from torchmetrics.classification import BinaryAccuracy

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter)

# To filter UserWarning.
warnings.filterwarnings("ignore", category=UserWarning)

bold = f"\033[1m"
reset = f"\033[0m"

In [29]:
block_plot=False

## 2 System Config

In [30]:
def system_config(SEED_VALUE=42):
    """
    Configures the system environment for PyTorch-based operations.

    Args:
        SEED_VALUE (int): Seed value for random number generation. Default is 42.

    Returns:
        tuple: A tuple containing the device name as a string and a boolean indicating GPU availability.
    """

    random.seed(SEED_VALUE)
    np.random.seed(SEED_VALUE)
    torch.manual_seed(SEED_VALUE)

    print("Using CPU")
    DEVICE = torch.device("cpu")
    print("Device: ", DEVICE)
    GPU_AVAILABLE = False

    torch.use_deterministic_algorithms(True)

    return str(DEVICE), GPU_AVAILABLE

In [31]:
DEVICE, GPU_AVAILABLE = system_config()

Using CPU
Device:  cpu


## 3 Preprocessing Transforms

Before we forward pass the image data through the model, it needs to accept the data in batches as PyTorch tensors, pre-processed through a fixed image resolution. The following functions enables us to pre-process our data before it is passed through the CNN model.

* `image_preprocess_transforms()`: Resizes the image data into fixed image resolution, and converts PIL images to PyTorch tensors (and optionally scaling them if they are of `uint8` images).

* `get_common_transforms()`: The transformed images obtained from `image_preprocess_transforms` are then normalized using the appropriate mean and standard deviation.

### 3.1 Preporocess Transforms

In [32]:
def image_preprocess_transforms(img_size):

    preprocess = T.Compose([T.Resize((img_size[0], img_size[1]), antialias=True),
                            T.ToTensor()
                           ])
    return preprocess

### 3.2 Common Transforms

In [33]:
def get_common_transforms(resize_to, mean, std):

    preprocess = image_preprocess_transforms(img_size=resize_to)

    common_transforms = T.Compose([
        # Apply the usual pre-processing transforms.
        preprocess,

        # Normalize the image by subtracting the mean and dividing by the std dev. of the training dataset.
        T.Normalize(mean, std)
    ])

    return common_transforms

## 4 Create Test Dataloader

The `get_data` function is used to create the test dataloader.

In [34]:
def get_data(data_root, 
             img_size,
             batch_size,
             mean=(0.5,0.5,0.5),
             std=(0.5,0.5,0.5),
             shuffle_val_data=False,
             num_workers=1):

    test_data_path = os.path.join(data_root, "test")

    common_transforms = get_common_transforms(resize_to=img_size,
                                              mean=mean,
                                              std=std)


    # Shuffled Test Dataset.
    test_dataset = datasets.ImageFolder(root=test_data_path, transform=common_transforms)
    # Shuffle Indices.
    shuffled_idx = torch.randperm(len(test_dataset))
    # Get shuffled dataset
    test_dataset_shuffled = Subset(test_dataset, indices=shuffled_idx)
    # Test DataLoader.
    test_loader = DataLoader(test_dataset_shuffled,
                             batch_size=batch_size,
                             shuffle=shuffle_val_data,
                             num_workers=num_workers
                            )

    return test_loader

## 5 Configuration [20 Points]

All training parameters are defined here. So, this is where you can improve your accuracy, apart from improving the architecture.

Here are a few hints on how you can improve the accuracy:
- Train for a larger number of epochs
- Try different learning rates
- Try with different image resolution

**You need to achieve atleast 92% accuracy on this given data to obtain atleast 10 points. To obtain the complete 20 points, you need to achieve atleast 95% for this assignment.**

<font style="color:red;font-size:15px"><b>Note:</b> You need to copy-paste the mean and std of the training data that was used during training.<br>Also ensure that you are using the same <b>image resolution</b> used during training.</font>

A sample snapshot is shown below.<br><br>
<img src="https://opencv.org/wp-content/uploads/2024/02/cnn-assignment-mean-std.png" width=700>

In [35]:
@dataclass(frozen=True)
class DatasetConfig:
    BATCH_SIZE:  int = 8 # Please don't change the batch size here.
    IMG_HEIGHT:  int = 224
    IMG_WIDTH:   int = 224
    MEAN:      tuple = (0.6501, 0.5935, 0.5400) # Pass the computed mean and std from the training notebook
    STD:       tuple = (0.2951, 0.3002, 0.3204)
    NUM_WORKERS: int = 4
    DATA_ROOT:   str = "./data/data-muffin-vs-chihuahua"
    
    # You are entitled to modify the above parameters except NUM_WORKERS, DATA_ROOT.
    ###
    ###

@dataclass(frozen=True)
class TrainingConfig:
    EPOCHS:          int  = 5
    LEARNING_RATE:  float = 1e-2
    MOMENTUM:       float = 0.9
    DEVICE:         str = "cpu"
    CHECKPOINT_DIR: str = "CHECKPOINTS"
    MODEL_FILENAME: str = "Model_best_acc.pth"

    # You can modify the above parameters except CHECKPOINT_DIR and MODEL_FILENAME.
    ###
    ###


Let us now initialize the parameters defined above.

In [36]:
train_config = TrainingConfig(
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu",
    EPOCHS = 70,                 
    LEARNING_RATE = 3e-4,        
)
dataset_config = DatasetConfig()


## 6 Model Definition [10 Points]

You need to write the model architecture here.

**Note:**

1. Since we will be using **Binary Cross Entropy** as the loss function, you need to ensure the correct `out_features` for the final fully connected (linear) layer.
2. You need to make sure that the model parameters are not too high. We were able to achieve well over **96.5%** test accuracy with the number of parameters as low as **1.7M**.

Here are a few pointers that you can implement in case you don't get the desired accuracy:
1. Try adding more convolution layers to the backbone.
2. Experiment with the kernel size in the Convolution layers.
3. Try adding an Adaptive Average Pooling Layer as the final layer in the backbone.
4. Try increasing the filters/nodes in the Conv/Linear layers.

In [37]:
class BinaryClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),   


            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), 
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1),    
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x        


Let us now print the model summary.

In [38]:
model = BinaryClassifierModel()

print(summary(model,
              input_size=(1, 3, dataset_config.IMG_HEIGHT, dataset_config.IMG_WIDTH),
              device="cpu",
              row_settings=["var_names"]))

Layer (type (var_name))                            Output Shape              Param #
BinaryClassifierModel (BinaryClassifierModel)      [1, 1]                    --
├─Sequential (features)                            [1, 512, 1, 1]            --
│    └─Conv2d (0)                                  [1, 32, 224, 224]         896
│    └─BatchNorm2d (1)                             [1, 32, 224, 224]         64
│    └─ReLU (2)                                    [1, 32, 224, 224]         --
│    └─MaxPool2d (3)                               [1, 32, 112, 112]         --
│    └─Conv2d (4)                                  [1, 64, 112, 112]         18,496
│    └─BatchNorm2d (5)                             [1, 64, 112, 112]         128
│    └─ReLU (6)                                    [1, 64, 112, 112]         --
│    └─MaxPool2d (7)                               [1, 64, 56, 56]           --
│    └─Conv2d (8)                                  [1, 128, 56, 56]          73,856
│    └─BatchNorm2d (9)   

In [39]:
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms as T

def get_train_val_loaders(
    data_root: str,
    img_size: tuple,
    batch_size: int,
    mean: tuple,
    std: tuple,
    val_split: float = 0.15,
    num_workers: int = 2,
    seed: int = 42,
):
    train_dir = os.path.join(data_root, "train")

    train_tfms = T.Compose([
        T.RandomResizedCrop((img_size[0], img_size[1]), antialias=True),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])

    val_tfms = T.Compose([
        T.Resize((img_size[0], img_size[1]), antialias=True),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])

    full_train = datasets.ImageFolder(root=train_dir, transform=train_tfms)
    n_total = len(full_train)
    n_val = int(n_total * val_split)
    n_train = n_total - n_val

    g = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(full_train, [n_train, n_val], generator=g)

    # Important: val must use val transforms
    val_subset.dataset = datasets.ImageFolder(root=train_dir, transform=val_tfms)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader


In [40]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_criterion_optimizer_scheduler(model):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=train_config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    return criterion, optimizer, scheduler

def train_one_epoch(model, loader, criterion, optimizer, device="cpu", scaler=None):
    model.train()
    acc_metric = BinaryAccuracy().to(device)
    running_loss = 0.0

    for imgs, targets in loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        targets_float = targets.float()

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(dim=-1)
                loss = criterion(logits, targets_float)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs).squeeze(dim=-1)
            loss = criterion(logits, targets_float)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = torch.round(torch.sigmoid(logits))
        acc_metric.update(preds, targets)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = acc_metric.compute().item()
    return epoch_loss, epoch_acc

@torch.no_grad()
def validate(model, loader, criterion, device="cpu"):
    model.eval()
    acc_metric = BinaryAccuracy().to(device)
    running_loss = 0.0

    for imgs, targets in loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        targets_float = targets.float()
        logits = model(imgs).squeeze(dim=-1)
        loss = criterion(logits, targets_float)
        running_loss += loss.item() * imgs.size(0)
        preds = torch.round(torch.sigmoid(logits))
        acc_metric.update(preds, targets)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = acc_metric.compute().item()
    return epoch_loss, epoch_acc

def save_best(model, acc: float, ckpt_dir=TrainingConfig.CHECKPOINT_DIR, filename=TrainingConfig.MODEL_FILENAME):
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(ckpt_dir, filename))


In [41]:
try:
    set_seed(42)
    torch.use_deterministic_algorithms(False)
except AttributeError:
    pass


train_loader, val_loader = get_train_val_loaders(
    data_root=dataset_config.DATA_ROOT,
    img_size=(dataset_config.IMG_HEIGHT, dataset_config.IMG_WIDTH),
    batch_size=dataset_config.BATCH_SIZE,
    mean=dataset_config.MEAN,
    std=dataset_config.STD,
    val_split=0.15,
    num_workers=dataset_config.NUM_WORKERS,
    seed=42
)

model = BinaryClassifierModel().to(train_config.DEVICE)

criterion, optimizer, scheduler = get_criterion_optimizer_scheduler(model)
scaler = torch.cuda.amp.GradScaler(enabled=(train_config.DEVICE=="cuda"))

best_acc = -1.0
for epoch in range(1, train_config.EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device=train_config.DEVICE, scaler=scaler)
    val_loss, val_acc = validate(model, val_loader, criterion, device=train_config.DEVICE)

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch
        save_best(model, best_acc)

    scheduler.step(val_acc)

    print(f"Epoch {epoch:02d}/{train_config.EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f} | best {best_acc:.4f} | {time.time()-t0:.1f}s")

print("\n" + "="*60)
print(f"\n Training complete! Best model was from epoch {best_epoch}")
print(f"Best validation accuracy: {best_acc:.4f}")

trained_model = BinaryClassifierModel().to(train_config.DEVICE)
trained_model.load_state_dict(torch.load(os.path.join(TrainingConfig.CHECKPOINT_DIR, TrainingConfig.MODEL_FILENAME),
                                         map_location=train_config.DEVICE))


/tmp/ipykernel_84759/2329490040.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(train_config.DEVICE=="cuda"))
/tmp/ipykernel_84759/553812849.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 01/70 | train loss 0.5388 acc 0.7420 | val loss 0.4143 acc 0.8110 | best 0.8110 | 6.7s
Epoch 02/70 | train loss 0.4665 acc 0.7900 | val loss 0.3819 acc 0.8519 | best 0.8519 | 6.9s
Epoch 03/70 | train loss 0.4628 acc 0.7982 | val loss 0.3142 acc 0.8604 | best 0.8604 | 6.9s
Epoch 04/70 | train loss 0.4393 acc 0.8091 | val loss 0.3713 acc 0.8364 | best 0.8604 | 6.7s
Epoch 05/70 | train loss 0.4349 acc 0.8168 | val loss 0.2953 acc 0.8731 | best 0.8731 | 6.9s
Epoch 06/70 | train loss 0.4129 acc 0.8265 | val loss 0.2815 acc 0.8773 | best 0.8773 | 7.0s
Epoch 07/70 | train loss 0.4251 acc 0.8196 | val loss 0.3108 acc 0.8787 | best 0.8787 | 6.8s
Epoch 08/70 | train loss 0.4036 acc 0.8315 | val loss 0.3883 acc 0.8322 | best 0.8787 | 7.1s
Epoch 09/70 | train loss 0.3927 acc 0.8328 | val loss 0.2694 acc 0.8928 | best 0.8928 | 6.8s
Epoch 10/70 | train loss 0.3949 acc 0.8342 | val loss 0.3420 acc 0.8561 | best 0.8928 | 6.8s
Epoch 11/70 | train loss 0.3727 acc 0.8462 | val loss 0.3662 acc 0.868

<All keys matched successfully>

In [42]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###


In [43]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###


In [44]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###


In [45]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###


## 7 Evaluation Step

In [46]:
def evaluate(
    train_config: TrainingConfig,
    model: nn.Module,
    test_loader: torch.utils.data.DataLoader,
) -> Tuple[float, float]:

    # Model in eval() mode
    model.eval()

    acc_metric = BinaryAccuracy()
    mean_metric = MeanMetric()

    total_steps = len(test_loader)


#     prog_bar = tqdm(test_loader, bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
    
    begin_t = time.time()

    for idx, (data, targets) in enumerate(test_loader,1):
        
        clear_output(wait=True)

        # Send data and target to the appropriate device.
        batch_data = data.to(train_config.DEVICE)
        batch_targets = targets.float().to(train_config.DEVICE)

        # Get the model's predicted logits.
        with torch.no_grad():
            logits = model(batch_data).squeeze(dim=-1)

        # Compute the CE-Loss.
        test_loss = F.binary_cross_entropy_with_logits(logits, batch_targets).item()

        # Get the batch predictions after converting them to probability scores.
        pred_idx = torch.round(logits.sigmoid())

        # Batch validation loss.
        batch_loss = mean_metric(test_loss, weight=data.shape[0])

        # Batch validation accuracy.
        _ = acc_metric(pred_idx.cpu(), targets)

        # Update progress bar description.
        step_status = f"Step: {bold}{idx}/{total_steps}{reset}"
        step_status+= f" Loss: {mean_metric.compute():.4f}, Acc: {acc_metric.compute():.4f}"
        print(step_status)
        


    test_loss = mean_metric.compute()
    test_acc = acc_metric.compute()
    
    print(f"Test Loss: {test_loss:.3f}\tTest Acuracy: {test_acc:.2f}")
    
    print(f"\nTotal time taken: {(time.time()-begin_t):.2f}s")


    return test_loss, test_acc

In [47]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###


In [48]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###


## 8 Model Prediction on Test Data

### 8.1 Load Model

In [49]:
def load_model_from_checkpoint(ckpt_dir = TrainingConfig.CHECKPOINT_DIR,
                               model_filename = TrainingConfig.MODEL_FILENAME,
                               device = TrainingConfig.DEVICE):

    model_path = os.path.join(ckpt_dir, model_filename)
    model = BinaryClassifierModel()
    model.load_state_dict(torch.load(model_path, map_location=TrainingConfig.DEVICE))

    model = model.to(device)

    return model

### 8.2 Data Denormalization

Recall that the image data being forward propagated through the model is in normalized form. For visualization purpose, we need to de-normalize this image data. The denormalize function exactly does that.

In [50]:
def denormalize(img_tensor,
                mean = torch.tensor([0.5, 0.5, 0.5]),
                std = torch.tensor([0.5, 0.5, 0.5])):

    denorm_image = (img_tensor * std[:,None,None] + mean[:,None,None]).clamp(min=0., max=1.)

    return denorm_image

### 8.3 Perform Batch Prediction

In [51]:
def batch_prediction(model, image_batch, device="cpu"):

    # run model in evaluation mode.
    model.eval()

    with torch.no_grad():
        output_logits = model(image_batch.to(device)).squeeze(dim=-1)

    # Get probability score using softmax
    batch_probs = output_logits.sigmoid()

    # Get the predicted class IDs
    pred_cls_batch = torch.round(batch_probs)

    # Get masks for prob < 0.5 => Class = 0.
    masks = batch_probs < 0.5

    # Update prob for Class 0 to 1. prob
    batch_probs[masks] = 1.- batch_probs[masks]

    # Convert predictions to lists
    pred_cls_batch = pred_cls_batch.cpu().int().tolist()
    pred_prob_batch = batch_probs.cpu().tolist()

    return pred_cls_batch, pred_prob_batch

In [52]:
test_loader_eval = get_data(
    data_root=dataset_config.DATA_ROOT,
    img_size=(dataset_config.IMG_HEIGHT, dataset_config.IMG_WIDTH),
    batch_size=dataset_config.BATCH_SIZE,
    mean=dataset_config.MEAN,
    std=dataset_config.STD,
    shuffle_val_data=False,
    num_workers=dataset_config.NUM_WORKERS,
)

best_model_eval = load_model_from_checkpoint(device=train_config.DEVICE)
test_loss_eval, test_acc_eval = evaluate(train_config, best_model_eval, test_loader_eval)
print(f"Final Evaluation Accuracy: {float(test_acc_eval):.4f}")


Step: 148/148 Loss: 0.1540, Acc: 0.9409
Test Loss: 0.154	Test Acuracy: 0.94

Total time taken: 1.63s
Final Evaluation Accuracy: 0.9409


### 8.4 Plot Model Predictions

## 9 Conclusion

We train a binary classification model on a dataset consisting of chihuahuas and muffins. The distribution between these classes is similar, making training challenging. Although we have tried to achieve 95% test accuracy, we can improve it further. List down your findings on the significant changes you have implemented that helped your model achieve beyond 95%  test accuracy.